In [ ]:
# 1. Install the Anthropic SDK
!pip install -q anthropic tqdm

import json
import pandas as pd
from tqdm import tqdm
import anthropic
from google.colab import drive, userdata, files

In [ ]:
# 2. Mount Google Drive (if needed)
drive.mount('/content/drive')

In [ ]:
# 3. Setup Claude API
# IMPORTANT: In the Colab 'Secrets' (key icon), add a secret named
# ANTHROPIC_API_KEY with your key from https://console.anthropic.com
try:
    api_key = userdata.get('ANTHROPIC_API_KEY')
    client = anthropic.Anthropic(api_key=api_key)
    print("Claude API configured successfully.")
except Exception as e:
    print(f"Error: Make sure you added 'ANTHROPIC_API_KEY' to Colab Secrets. {e}")


In [ ]:
# 4. Query function — mirrors your query_gemini_agent()
def query_claude_agent(system_prompt, user_message):
    """Sends a request to Claude and returns parsed JSON."""
    try:
        response = client.messages.create(
            model="claude-sonnet-4-5",           # fast + smart; swap for claude-opus-4-5 if needed
            max_tokens=256,
            system=system_prompt,               # Claude separates system from user — cleaner than Gemini's concat
            messages=[
                {"role": "user", "content": user_message}
            ]
        )
        # Extract the text content from the response
        text = response.content[0].text.strip()
        return json.loads(text)
    except json.JSONDecodeError:
        print(f"JSON parse error. Raw response: {text}")
        return None
    except Exception as e:
        print(f"Error querying Claude: {e}")
        return None

In [ ]:
# 5. Agent prompts (unchanged from your original)

# --- Agent 1: Fact Checker ---
fact_system = """You are FactCheckerAgent.
Classify the review as: 'real', 'fake(misinformation)', or 'unrelated'.
Output ONLY raw JSON with no markdown formatting: {"label": "..."}"""

# --- Agent 2: Sentiment Analyzer ---
sentiment_system = """You are SentimentAgent.
Analyze sentiment as: 'positive', 'negative', or 'neutral'.
Output ONLY raw JSON with no markdown formatting: {"sentiment": "..."}"""

# --- Agent 3: Combiner ---
combiner_system = """You are CombinerAgent.
Review the provided labels and text. Correct any obvious logical errors.
Output ONLY raw JSON with no markdown formatting: {"label": "...", "sentiment": "..."}"""

In [ ]:
# 6. Load your JSONL data
data_path = "/content/drive/MyDrive/Czech/M12_final_v1_to_dataset.jsonl"

reviews = []
with open(data_path, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            try:
                reviews.append(json.loads(line))
            except:
                pass
        if len(reviews) >= 100:  # Testing cap — remove for full run
            break

print(f"Loaded {len(reviews)} reviews.")

In [ ]:
# 7. Run the 3-agent pipeline
results = []

for row in tqdm(reviews, desc="3-Agent Claude Processing"):
    text = row.get("text", "")

    if not text.strip():
        label, sentiment = "unrelated", "neutral"
    else:
        # Agent 1: Fact Check
        fact_res = query_claude_agent(fact_system, f"Text: {text}")
        label = fact_res.get("label", "unrelated") if fact_res else "unrelated"

        # Agent 2: Sentiment
        sent_res = query_claude_agent(sentiment_system, f"Text: {text}")
        sentiment = sent_res.get("sentiment", "neutral") if sent_res else "neutral"

        # Agent 3: Combiner
        combiner_input = f"Text: {text}\nLabel: {label}\nSentiment: {sentiment}"
        final_res = query_claude_agent(combiner_system, combiner_input)

        if final_res:
            label = final_res.get("label", label)
            sentiment = final_res.get("sentiment", sentiment)

    results.append({
        **row,
        "label": label,
        "sentiment": sentiment
    })

In [ ]:
# 8. Save and download
df = pd.DataFrame(results)
output_path = "/content/drive/MyDrive/Czech/claude-analyzed_reviews.csv"
df.to_csv(output_path, index=False, encoding="utf-8-sig")

print(f"Done! Saved {len(df)} rows to {output_path}")
files.download(output_path)